# Yazılım Kariyer Rehberi — Maaş Tahmini ve Alan Önerici

**Uçtan Uca Yapay Zeka Projesi — Software Persona Stajı**

**Problem:** Bir yazılımcının deneyim yılı, uzmanlık alanı, ülkesi, eğitim seviyesi, şirket büyüklüğü, çalışma şekli, sektörü, yaşı, unvanı (IC/PM), AI araç kullanımı ve bildiği dil sayısından yola çıkarak beklenen aylık maaşı (USD ve TL) tahmin etmek; ayrıca kişisel tercihlerine göre 9 yazılım alanından en uygun olanı önermek.

**Veri Seti:** Stack Overflow Developer Survey 2025 (Kaggle) — 49.123 geliştiricinin yanıtları

**Kapsam:** Veri temizleme → Keşifçi Veri Analizi → Model Karşılaştırma → Hiperparametre Optimizasyonu → Değerlendirme → Alan Önerici Mini Uygulama → Modeli Dışa Aktarma (deploy için)

## Kurulum

In [ ]:
# --- Gerekli kütüphaneleri yükle ---
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

try:
    import xgboost as xgb
except ImportError:
    %pip install xgboost --quiet
    import xgboost as xgb

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Kurulum tamamlandı. Rastgelelik sabiti:', RANDOM_STATE)

## Aşama 2: Veri Toplama ve Hazırlama

Notebook önce yanındaki dosyaları arar (yerelden ya da Colab'a yüklenmiş halinden okur). Bulamazsa Kaggle API'siyle otomatik indirmeyi dener. Böylece Colab oturumu kapanıp yeniden açıldığında dosyaları tekrar tekrar elle yüklemek gerekmez.

In [ ]:
# --- Kaggle Verisini Hazırla ---
# Not: Kaggle indirme bir hesap/API anahtarı gerektirdiği için otomatik indirme
# her ortamda çalışmayabilir. Önce dosyanın olası konumlarına bakıyoruz;
# yoksa kaggle API/CLI ile indirmeyi deniyoruz (Kaggle API anahtarının
# ortamda tanımlı olması gerekir: ~/.kaggle/kaggle.json).

DOSYA_ADI = 'survey_results_public.csv'

adaylar = [
    Path.cwd() / DOSYA_ADI,
    Path.cwd() / 'dataset' / DOSYA_ADI,
    Path('/content') / DOSYA_ADI,
    Path('/content/dataset') / DOSYA_ADI,
]

veri_yolu = next((yol for yol in adaylar if yol.exists()), None)

if veri_yolu is None:
    print('Dosya yerelde bulunamadı, Kaggle API ile indirilmeye çalışılacak...')
    print("(Kaggle API anahtarı kurulu değilse, dosyayı elle Colab'a yükleyip")
    print(' bu hücreyi tekrar çalıştırman yeterli.)')
    try:
        import kaggle
        hedef_klasor = Path('/content/dataset') if Path('/content').exists() else Path.cwd() / 'dataset'
        hedef_klasor.mkdir(parents=True, exist_ok=True)
        kaggle.api.dataset_download_files(
            'edoardogalli/stack-overflow-annual-developer-survey-2025',
            path=str(hedef_klasor), unzip=True
        )
        veri_yolu = hedef_klasor / DOSYA_ADI
    except Exception as hata:
        print('Otomatik indirme başarısız oldu:', hata)
        print("Lütfen survey_results_public.csv dosyasını Colab'a manuel yükle.")

print('Kullanılan veri:', veri_yolu)

df = pd.read_csv(veri_yolu)
print('Ham veri boyutu:', df.shape)

In [ ]:
# --- Projede kullanılacak sütunları seç (genişletilmiş özellik seti) ---
# WorkExp: profesyonel iş deneyimi (yıl)
# DevType: geliştirici rolü/uzmanlığı
# Country: ülke
# EdLevel: eğitim seviyesi
# RemoteWork: çalışma şekli (uzaktan/hibrit/ofis)
# OrgSize: şirket büyüklüğü
# Industry: sektör
# Employment: istihdam tipi
# Age: yaş aralığı
# ICorPM: yönetici mi bireysel katkıcı mı
# AISelect: AI araç kullanım sıklığı
# LanguageHaveWorkedWith: bilinen programlama dilleri (';' ile ayrılmış) -> dil sayısına çevrilecek
# ConvertedCompYearly: yıllık maaş (USD'ye çevrilmiş) — hedef değişkenimizin kaynağı

cols_extended = ['WorkExp', 'DevType', 'Country', 'EdLevel',
                 'RemoteWork', 'OrgSize', 'Industry', 'Employment', 'Age',
                 'ICorPM', 'AISelect', 'LanguageHaveWorkedWith',
                 'ConvertedCompYearly']

df_raw = df[cols_extended].copy()
df_raw = df_raw.dropna(subset=cols_extended)

# --- Gerçekçi olmayan uç değerleri filtrele ---
df_raw = df_raw[(df_raw['ConvertedCompYearly'] >= 1000) & (df_raw['ConvertedCompYearly'] <= 500000)]
df_raw = df_raw[df_raw['WorkExp'] <= 50]

# --- Dil sayısını türet ---
df_raw['DilSayisi'] = df_raw['LanguageHaveWorkedWith'].apply(lambda x: len(str(x).split(';')))

# --- Aylık maaş sütununu türet ---
df_clean = df_raw.copy()
df_clean['MonthlySalaryUSD'] = df_clean['ConvertedCompYearly'] / 12

print('Temizlenmiş veri boyutu:', df_clean.shape)

### Veri Sözlüğü

| Sütun | Tipi | Anlamı |
|---|---|---|
| WorkExp | Sayısal | Profesyonel iş deneyimi (yıl) |
| DevType | Kategorik | Geliştirici rolü/uzmanlığı |
| Country | Kategorik | Yaşadığı ülke |
| EdLevel | Kategorik | Eğitim seviyesi |
| RemoteWork | Kategorik | Çalışma şekli (uzaktan/hibrit/ofis) |
| OrgSize | Kategorik | Şirket büyüklüğü |
| Industry | Kategorik | Çalıştığı sektör |
| Employment | Kategorik | İstihdam tipi |
| Age | Kategorik | Yaş aralığı |
| ICorPM | Kategorik | Bireysel katkıcı mı, yönetici mi |
| AISelect | Kategorik | AI araç kullanım sıklığı |
| DilSayisi | Sayısal | Bilinen programlama dili sayısı (LanguageHaveWorkedWith'ten türetildi) |
| MonthlySalaryUSD | Sayısal | Hedef değişken — aylık maaş (USD) |

**Temizleme Adımları:** Eksik veri içeren satırlar atıldı; maaş 1.000-500.000 USD/yıl aralığı dışındaki uç değerler ve 50 yıldan fazla deneyim beyan eden satırlar filtrelendi. Sonuç: 18.223 temiz satır.

## Aşama 3: Keşifçi Veri Analizi (EDA) ve Görselleştirme

In [ ]:
# --- Genel özet ---
print(df_clean.describe())
print(df_clean['DevType'].value_counts().head(10))
print(df_clean['Country'].value_counts().head(10))

In [ ]:
# --- Grafik 1: Deneyim yılına göre ortalama aylık maaş ---
# 2 yıllık gruplara ayırarak trendi net göstermek amacıyla ortalama çizgisi kullanıyoruz
# (ham dağılım/scatter grafiği çok sayıda nokta yüzünden yığılma gösterdiği için tercih edilmedi)

df_clean['ExpGroup'] = (df_clean['WorkExp'] // 2) * 2
avg_by_exp = df_clean.groupby('ExpGroup')['MonthlySalaryUSD'].mean()

plt.figure(figsize=(10, 6))
plt.plot(avg_by_exp.index, avg_by_exp.values, marker='o', color='steelblue')
plt.title('Deneyim Yılına Göre Ortalama Aylık Maaş (USD)')
plt.xlabel('Profesyonel Deneyim (Yıl)')
plt.ylabel('Ortalama Aylık Maaş (USD)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Grafik 2: DevOps mühendislerinde deneyime göre ortalama aylık maaş ---

df_devops = df_clean[df_clean['DevType'] == 'DevOps engineer or professional'].copy()
print('DevOps mühendisi sayısı:', df_devops.shape[0])

avg_devops = df_devops.groupby('ExpGroup')['MonthlySalaryUSD'].mean()

plt.figure(figsize=(10, 6))
plt.plot(avg_devops.index, avg_devops.values, marker='o', color='darkorange')
plt.title('DevOps Mühendislerinde Deneyime Göre Ortalama Aylık Maaş (USD)')
plt.xlabel('Profesyonel Deneyim (Yıl)')
plt.ylabel('Ortalama Aylık Maaş (USD)')
plt.grid(alpha=0.3)
plt.show()

### Keşif Bulguları

1. Deneyimin ilk 10 yılında maaşta hızlı ve net bir artış görülüyor.
2. 10-30 yıl arası artış devam ediyor ama daha yavaş bir tempoda.
3. 30 yıl üstü deneyimde dalgalanma artıyor — bu, o deneyim seviyelerindeki örnek sayısının azlığından kaynaklanan istatistiksel gürültü olabilir.
4. DevOps mühendisleri (672 kişi) için de benzer bir eğilim var: ilk yıllarda hızlı artış, sonra yavaşlayan bir yükseliş.
5. Genel örneklemde en çok temsil edilen roller ve ülkeler, veri setinin ABD ve Batı Avrupa ağırlıklı olduğunu gösteriyor — bu, modelin bazı bölgeler için daha az güvenilir tahmin üretebileceği anlamına gelir.

## Aşama 4: Model Kurma, Karşılaştırma ve Optimizasyon

In [ ]:
# --- Özellik (X) ve hedef (y) ayır ---
X = df_clean[['WorkExp', 'DevType', 'Country', 'EdLevel',
              'RemoteWork', 'OrgSize', 'Industry', 'Employment', 'Age',
              'ICorPM', 'AISelect', 'DilSayisi']]
y = df_clean['MonthlySalaryUSD']

categorical_cols = ['DevType', 'Country', 'EdLevel', 'RemoteWork', 'OrgSize',
                     'Industry', 'Employment', 'Age', 'ICorPM', 'AISelect']
# WorkExp ve DilSayisi sayısal olarak kalıyor (remainder='passthrough')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Eğitim seti boyutu:', X_train.shape)
print('Test seti boyutu:', X_test.shape)

### Model Karşılaştırması

Dört farklı regresyon modeli deneyip performanslarını karşılaştırıyoruz: basit bir doğrusal model (LinearRegression) ile başlayıp, doğrusal olmayan ilişkileri de yakalayabilen ağaç tabanlı modellere (RandomForest, GradientBoosting, XGBoost) geçiyoruz.

In [ ]:
sonuclar = {}

# --- Model 1: LinearRegression ---
model_lr = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)
sonuclar['LinearRegression'] = (r2_score(y_test, pred_lr), mean_absolute_error(y_test, pred_lr))

# --- Model 2: RandomForestRegressor ---
model_rf = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(
    n_estimators=300, max_depth=None, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
))])
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)
sonuclar['RandomForest'] = (r2_score(y_test, pred_rf), mean_absolute_error(y_test, pred_rf))

# --- Model 3: GradientBoostingRegressor ---
model_gb = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE
))])
model_gb.fit(X_train, y_train)
pred_gb = model_gb.predict(X_test)
sonuclar['GradientBoosting'] = (r2_score(y_test, pred_gb), mean_absolute_error(y_test, pred_gb))

print('İlk 3 model eğitildi:')
for isim, (r2, mae) in sonuclar.items():
    print(f'  {isim}: R²={r2:.3f}, MAE={mae:.2f} USD/ay')

### Hiperparametre Optimizasyonu (XGBoost + GridSearch)

En iyi sonucu almak için XGBoost'u GridSearch ile optimize ediyoruz — bu, birden fazla hiperparametre kombinasyonunu çapraz doğrulamayla dener ve en iyisini seçer.

In [ ]:
# --- Model 4: XGBoost + GridSearch ---
model_xgb_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    'regressor__n_estimators': [200, 300, 400],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [3, 4, 5],
}

grid_search = GridSearchCV(
    model_xgb_base, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)

print('GridSearch başlıyor, birkaç dakika sürebilir...')
grid_search.fit(X_train, y_train)

print('\nEn iyi parametreler:', grid_search.best_params_)
print('En iyi çapraz doğrulama R²:', grid_search.best_score_)

# --- Final model: GridSearch'ün bulduğu en iyi model ---
model = grid_search.best_estimator_
y_pred = model.predict(X_test)

sonuclar['XGBoost (optimize)'] = (r2_score(y_test, y_pred), mean_absolute_error(y_test, y_pred))

print('\n=== TÜM MODELLERİN KARŞILAŞTIRMASI ===')
for isim, (r2, mae) in sonuclar.items():
    print(f'  {isim}: R²={r2:.3f}, MAE={mae:.2f} USD/ay')

print('\n>>> Final model olarak XGBoost (optimize edilmiş) seçildi. <<<')

**Model Seçim Gerekçesi:** Dört model karşılaştırıldı: basit LinearRegression bir başlangıç referansı sağladı; RandomForest ve GradientBoosting doğrusal olmayan ilişkileri yakalamayı denedi; GridSearch ile optimize edilmiş XGBoost en iyi sonucu verdi (en yüksek R², en düşük MAE). Bu yüzden final model olarak XGBoost seçildi. Deep learning (sinir ağları) bilinçli olarak tercih edilmedi çünkü: (1) veri seti boyutu (~18.000 satır) bir sinir ağını beslemek için görece küçük, (2) veri tablo (tabular) formatında olduğu için ağaç tabanlı modeller literatürde genelde sinir ağlarına eşit veya üstün performans gösteriyor, daha az veriyle ve daha az eğitim süresiyle.

## Aşama 5: Tahmin ve Değerlendirme

In [ ]:
print(f'Final Model (XGBoost) — R²: {r2_score(y_test, y_pred):.3f}')
print(f'Final Model (XGBoost) — MAE: {mean_absolute_error(y_test, y_pred):.2f} USD/ay')

In [ ]:
# --- 3 yeni örnek profil için tahmin üret ---
USD_TO_TRY = 40  # sabit kur varsayımı (rapora not düşülmüştür)

yeni_ornekler = pd.DataFrame([
    {'WorkExp': 2, 'DevType': 'DevOps engineer or professional', 'Country': 'Turkey',
     'EdLevel': 'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)',
     'RemoteWork': 'Remote', 'OrgSize': '20 to 99 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '25-34 years old',
     'ICorPM': 'Individual contributor', 'AISelect': 'Yes, I use AI tools weekly', 'DilSayisi': 4},

    {'WorkExp': 8, 'DevType': 'DevOps engineer or professional', 'Country': 'Turkey',
     'EdLevel': 'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)',
     'RemoteWork': 'Remote', 'OrgSize': '100 to 499 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '25-34 years old',
     'ICorPM': 'Individual contributor', 'AISelect': 'Yes, I use AI tools daily', 'DilSayisi': 7},

    {'WorkExp': 15, 'DevType': 'Developer, back-end', 'Country': 'United States of America',
     'EdLevel': 'Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)',
     'RemoteWork': 'Hybrid (some in-person, leans heavy to flexibility)', 'OrgSize': '1,000 to 4,999 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '35-44 years old',
     'ICorPM': 'People manager', 'AISelect': 'Yes, I use AI tools daily', 'DilSayisi': 9},
])

tahminler = model.predict(yeni_ornekler)

for i, tahmin in enumerate(tahminler):
    tahmin_tl = tahmin * USD_TO_TRY
    print(f'Örnek {i+1}: Tahmini Aylık Maaş = {tahmin:.2f} USD (~{tahmin_tl:,.0f} TL)')

### Değerlendirme Notu

Final model (XGBoost, optimize edilmiş), maaştaki değişimin önemli bir kısmını açıklayabiliyor ve ilk basit modele (R²=0.468) göre belirgin bir iyileşme sağladı. Bu sonuç gerçekçi: maaşı etkileyen pek çok faktör (spesifik teknoloji becerileri, pazarlık gücü, şirketin finansal durumu vb.) veri setinde yer almıyor. Model aşırı öğrenme riski taşımıyor çünkü performans, eğitimde hiç görülmemiş bir test seti üzerinde ölçüldü ve GridSearch çapraz doğrulama kullanarak hiperparametreleri seçti.

### Modelin Sinyallerini İncele (GradientBoosting üzerinden yorumlama)

XGBoost'un ham katsayı yorumu doğrusal modeller kadar basit olmadığı için, hangi özelliklerin maaşı yukarı/aşağı çektiğini görmek adına GradientBoosting modelinin özellik önemini (feature importance) inceliyoruz.

In [ ]:
# --- GradientBoosting modelinin özellik önemini görselleştir ---
ozellik_adlari = model_gb.named_steps['preprocessor'].get_feature_names_out()
onemler = pd.Series(
    model_gb.named_steps['regressor'].feature_importances_,
    index=ozellik_adlari
)

onemler.index = (
    onemler.index.str.replace('cat__', '', regex=False)
    .str.replace('remainder__', '', regex=False)
)

en_onemli_10 = onemler.nlargest(10).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(en_onemli_10.index, en_onemli_10.values, color='#1F3564')
plt.title('Maaş Tahmininde En Etkili İlk 10 Özellik')
plt.xlabel('Özellik Önemi')
plt.tight_layout()
plt.show()

en_onemli_10.round(4)

**Not:** Özellik önemi, bir özelliğin modelin tahminlerinde ne kadar sıklıkla ve ne kadar etkili kullanıldığını gösterir; tek başına nedensellik ifade etmez (örn. "ABD'de olmak" tek başına maaşı artırmaz, piyasa koşullarıyla ilişkilidir). Yine de bu grafik, modelin hangi faktörlere daha çok ağırlık verdiğini şeffaf bir şekilde gösteriyor.

## Aşama 6: Akıllı Tahmin Aracı — Genişletilmiş Alan Önerici

9 yazılım alanı arasından, 15 soruluk 4 şıklı bir anketle kullanıcının profiline en uygun olanı önerir. Alanlar arası adil karşılaştırma için puanlar, her alanın alabileceği maksimum puana göre normalize edilir (yüzdeye çevrilir) — aksi halde bazı alanlar (örn. Game Development) yapısal olarak diğerlerinden (örn. Data Science) daha düşük maksimum puana sahip olup haksız yere geride kalabiliyordu.

Önerilen alana göre, kullanıcıya **roadmap.sh** üzerindeki ilgili resmi öğrenme yol haritasının linki de sunuluyor — böylece uygulama sadece "sana bu alan uygun" demekle kalmıyor, "bu alanda ne öğrenmen gerektiğinin adım adım haritası burada" diyerek somut bir sonraki adım sağlıyor.

In [ ]:
def alan_oner():
    """9 farklı yazılım alanı arasından, 15 soruluk 4 şıklı bir anketle
    kullanıcının profiline en uygun olanı önerir. Puanlar, her alanın
    alabileceği maksimum puana göre normalize edilir (adil karşılaştırma için).
    Önerilen alana göre roadmap.sh üzerinden bir öğrenme yol haritası linki de sunar."""

    puanlar = {
        'Backend': 0, 'Frontend': 0, 'Mobile': 0, 'DevOps/Cloud': 0,
        'Data Science/AI-ML': 0, 'QA/Test': 0, 'Cybersecurity': 0,
        'Game Development': 0, 'Embedded/IoT': 0,
    }

    # --- Her alan için roadmap.sh üzerindeki resmi yol haritası linki ---
    # Not: roadmap.sh'de "Mobile" ve "Embedded/IoT" için tek bir resmi sayfa yok;
    # Mobile için Android roadmap'i, Embedded/IoT için en yakın genel kaynak verildi.
    roadmap_linkleri = {
        'Backend': 'https://roadmap.sh/backend',
        'Frontend': 'https://roadmap.sh/frontend',
        'Mobile': 'https://roadmap.sh/android',
        'DevOps/Cloud': 'https://roadmap.sh/devops',
        'Data Science/AI-ML': 'https://roadmap.sh/ai-data-scientist',
        'QA/Test': 'https://roadmap.sh/qa',
        'Cybersecurity': 'https://roadmap.sh/cyber-security',
        'Game Development': 'https://roadmap.sh/game-developer',
        'Embedded/IoT': 'https://roadmap.sh/computer-science',
    }

    sorular = [
        {'soru': "1) Bir projede en çok hangisiyle vakit geçirmek isterdin?",
         'secenekler': {
            '1': ("Sunucu tarafı iş mantığı ve API'ler kurmak", {'Backend': 3, 'DevOps/Cloud': 1}),
            '2': ("Kullanıcının gördüğü ekranları tasarlamak", {'Frontend': 3, 'Mobile': 1}),
            '3': ("Veriden model kurup tahmin üretmek", {'Data Science/AI-ML': 3}),
            '4': ("Bir oyunun mekaniklerini kodlamak", {'Game Development': 3}),
         }},
        {'soru': "2) Hangi ortamda çalışmak sana daha cazip gelir?",
         'secenekler': {
            '1': ("Bulut altyapısı, sunucular, otomasyon boru hatları", {'DevOps/Cloud': 3, 'Backend': 1}),
            '2': ("Telefon/tablet uygulamaları geliştirme ortamı", {'Mobile': 3}),
            '3': ("Fiziksel donanım ve sensörlerle entegre sistemler", {'Embedded/IoT': 3}),
            '4': ("Sistemleri saldırılara karşı test etme/koruma ortamı", {'Cybersecurity': 3}),
         }},
        {'soru': "3) Bir yazılım hatasıyla karşılaştığında ilk içgüdün nedir?",
         'secenekler': {
            '1': ("Sistematik olarak test senaryoları yazıp hatayı yakalarım", {'QA/Test': 3}),
            '2': ("Kodun mantığını satır satır takip edip kök nedeni bulurum", {'Backend': 2, 'DevOps/Cloud': 1}),
            '3': ("Kullanıcı arayüzünde nerede yanlış göründüğüne bakarım", {'Frontend': 2, 'Mobile': 1}),
            '4': ("Güvenlik açığı olup olmadığını kontrol ederim", {'Cybersecurity': 2, 'QA/Test': 1}),
         }},
        {'soru': "4) Hangi konuda kendini daha meraklı hissediyorsun?",
         'secenekler': {
            '1': ("Yapay zeka ve makine öğrenmesi modelleri", {'Data Science/AI-ML': 3}),
            '2': ("Oyun motorları ve interaktif deneyimler", {'Game Development': 3}),
            '3': ("IoT cihazları ve gömülü sistemler", {'Embedded/IoT': 3}),
            '4': ("Bulut mimarisi ve ölçeklenebilir sistemler", {'DevOps/Cloud': 3}),
         }},
        {'soru': "5) Bir ürün geliştirirken en çok neye özen gösterirsin?",
         'secenekler': {
            '1': ("Kodun performansı ve veri tabanı verimliliği", {'Backend': 3}),
            '2': ("Kullanıcı deneyiminin akıcı ve estetik olması", {'Frontend': 3, 'Mobile': 1}),
            '3': ("Her senaryonun test edilip hatasız çalışması", {'QA/Test': 3}),
            '4': ("Verinin güvenli ve gizli tutulması", {'Cybersecurity': 3}),
         }},
        {'soru': "6) Hangi çalışma tarzı sana daha uygun?",
         'secenekler': {
            '1': ("Derinlemesine, tek başıma analiz yaparak ilerlemek", {'Data Science/AI-ML': 2, 'Embedded/IoT': 1}),
            '2': ("Sürekli test-geliştir döngüsüyle hızlı ilerlemek", {'Game Development': 2, 'Frontend': 1}),
            '3': ("Sistemleri izleyip otomatikleştirerek ilerlemek", {'DevOps/Cloud': 2, 'Backend': 1}),
            '4': ("Detaylı senaryolar yazıp adım adım doğrulamak", {'QA/Test': 2, 'Cybersecurity': 1}),
         }},
        {'soru': "7) Aşağıdakilerden hangisi seni en çok heyecanlandırır?",
         'secenekler': {
            '1': ("Bir mobil uygulamanın milyonlarca kullanıcıya ulaşması", {'Mobile': 3}),
            '2': ("Bir modelin doğru tahminler üretmesi", {'Data Science/AI-ML': 3}),
            '3': ("Bir sistemin saldırıya karşı dayanıklı olması", {'Cybersecurity': 3}),
            '4': ("Bir oyunun akıcı ve eğlenceli olması", {'Game Development': 3}),
         }},
        {'soru': "8) Yeni bir teknoloji öğrenirken hangisi seni motive eder?",
         'secenekler': {
            '1': ("Sunucu tarafı yeni bir framework/API teknolojisi", {'Backend': 3}),
            '2': ("Yeni bir UI kütüphanesi veya tasarım aracı", {'Frontend': 3}),
            '3': ("Yeni bir test otomasyon aracı", {'QA/Test': 3}),
            '4': ("Yeni bir donanım/mikrodenetleyici platformu", {'Embedded/IoT': 3}),
         }},
        {'soru': "9) Bir sistemde en çok hangi soru ilgini çeker?",
         'secenekler': {
            '1': ("Bu sistem 1 milyon kullanıcıya nasıl ölçeklenir?", {'DevOps/Cloud': 3}),
            '2': ("Bu veri setinden ne tür bir örüntü çıkarabiliriz?", {'Data Science/AI-ML': 3}),
            '3': ("Bu uygulama tüm cihazlarda düzgün çalışıyor mu?", {'QA/Test': 2, 'Mobile': 1}),
            '4': ("Bu sistemde güvenlik açığı var mı?", {'Cybersecurity': 3}),
         }},
        {'soru': "10) Hangi proje türü sana daha cazip gelir?",
         'secenekler': {
            '1': ("Bir e-ticaret sitesinin arka uç (backend) sistemleri", {'Backend': 3}),
            '2': ("Bir mobil oyunun arayüzü ve mekanikleri", {'Game Development': 2, 'Mobile': 2}),
            '3': ("Akıllı ev cihazlarının yazılımı", {'Embedded/IoT': 3}),
            '4': ("Bir şirketin sunucularının bulutta yönetimi", {'DevOps/Cloud': 3}),
         }},
        {'soru': "11) Bir problemi çözerken hangi yaklaşımı benimsersin?",
         'secenekler': {
            '1': ("Veriyi analiz edip istatistiksel çıkarım yaparım", {'Data Science/AI-ML': 3}),
            '2': ("Olası tüm hata senaryolarını listeleyip test ederim", {'QA/Test': 3}),
            '3': ("Saldırgan gibi düşünüp zayıf noktaları ararım", {'Cybersecurity': 3}),
            '4': ("Kullanıcı gözünden deneyimi test ederim", {'Frontend': 3}),
         }},
        {'soru': "12) Hangi teknoloji alanı seni daha çok cezbediyor?",
         'secenekler': {
            '1': ("Docker, Kubernetes gibi konteyner/orkestasyon araçları", {'DevOps/Cloud': 3}),
            '2': ("React, Vue gibi arayüz kütüphaneleri", {'Frontend': 3}),
            '3': ("TensorFlow, PyTorch gibi makine öğrenmesi araçları", {'Data Science/AI-ML': 3}),
            '4': ("Arduino, Raspberry Pi gibi donanım platformları", {'Embedded/IoT': 3}),
         }},
        {'soru': "13) Uzun vadede kendini hangi rolde görüyorsun?",
         'secenekler': {
            '1': ("Sistem mimarisi kuran, arka planı yöneten biri", {'Backend': 3}),
            '2': ("Kullanıcı deneyimini şekillendiren biri", {'Mobile': 3}),
            '3': ("Veriyle karar destek sistemleri kuran biri", {'Data Science/AI-ML': 3}),
            '4': ("Sistemlerin güvenliğini sağlayan biri", {'Cybersecurity': 3}),
         }},
        {'soru': "14) Hangi görev sana daha keyifli gelir?",
         'secenekler': {
            '1': ("Bir uygulamanın performans testlerini yapmak", {'QA/Test': 3}),
            '2': ("Bir oyunun seviye tasarımını kodlamak", {'Game Development': 3}),
            '3': ("Bir sensörden gelen veriyi işleyen kod yazmak", {'Embedded/IoT': 3}),
            '4': ("Bir API'nin veritabanı sorgularını optimize etmek", {'Backend': 3}),
         }},
        {'soru': "15) Kariyerinde en çok neyle anılmak istersin?",
         'secenekler': {
            '1': ("Milyonlarca kullanıcının kullandığı bir mobil uygulama", {'Mobile': 3}),
            '2': ("Yenilikçi bir yapay zeka ürünü", {'Data Science/AI-ML': 3}),
            '3': ("Kusursuz test edilmiş, güvenilir bir sistem", {'QA/Test': 3}),
            '4': ("Büyük ölçekli, kesintisiz çalışan bulut altyapısı", {'DevOps/Cloud': 3}),
         }},
    ]

    alanlar = list(puanlar.keys())
    max_puan = {alan: 0 for alan in alanlar}
    for s in sorular:
        for alan in alanlar:
            en_iyi = max([secenek[1].get(alan, 0) for secenek in s['secenekler'].values()])
            max_puan[alan] += en_iyi

    for s in sorular:
        print(s['soru'])
        for numara, (metin, _) in s['secenekler'].items():
            print(f'   {numara}) {metin}')

        cevap = input('Cevabın (1/2/3/4): ').strip()

        if cevap in s['secenekler']:
            _, secilen_puanlar = s['secenekler'][cevap]
            for alan, puan in secilen_puanlar.items():
                puanlar[alan] += puan
        else:
            print('Geçersiz cevap, bu soru atlandı.')
        print()

    yuzdeler = {alan: round((puanlar[alan] / max_puan[alan]) * 100, 1) for alan in alanlar}

    siralanmis = sorted(yuzdeler.items(), key=lambda x: x[1], reverse=True)
    onerilen_alan = siralanmis[0][0]

    print('📊 Uyum Yüzdesi (yüksekten düşüğe):')
    for alan, yuzde in siralanmis:
        print(f'   {alan}: %{yuzde}  (ham puan: {puanlar[alan]}/{max_puan[alan]})')

    print(f"\n✅ Sana en uygun yazılım alanı: **{onerilen_alan}**")
    print(f'   (İkinci en yakın alan: {siralanmis[1][0]})')
    print(f'\n🗺️  {onerilen_alan} için önerilen öğrenme yol haritası (roadmap.sh):')
    print(f'   {roadmap_linkleri[onerilen_alan]}')

    return onerilen_alan, yuzdeler, roadmap_linkleri[onerilen_alan]


onerilen, tum_yuzdeler, roadmap_link = alan_oner()

## Aşama 7: Modeli Dışa Aktar (Deploy için)

Eğitilmiş maaş tahmin modelini bir dosyaya kaydediyoruz. Bu dosya, ayrı bir frontend/API projesinde (örn. FastAPI/Flask) yüklenip gerçek zamanlı tahmin üretmek için kullanılabilir.

In [ ]:
# --- Final modeli (XGBoost pipeline) diske kaydet ---
MODEL_DOSYA_ADI = 'maas_tahmin_modeli.joblib'

joblib.dump(model, MODEL_DOSYA_ADI)
print(f'Model kaydedildi: {MODEL_DOSYA_ADI}')

# --- Modelin beklediği sütun sırasını da kaydedelim (deploy tarafında işine yarar) ---
import json

model_bilgisi = {
    'ozellik_sirasi': list(X.columns),
    'kategorik_sutunlar': categorical_cols,
    'sayisal_sutunlar': ['WorkExp', 'DilSayisi'],
    'hedef_degisken': 'MonthlySalaryUSD (USD)',
    'usd_to_try_sabit_kur': USD_TO_TRY,
    'test_r2': round(r2_score(y_test, y_pred), 4),
    'test_mae': round(mean_absolute_error(y_test, y_pred), 2),
}

with open('model_bilgisi.json', 'w', encoding='utf-8') as f:
    json.dump(model_bilgisi, f, ensure_ascii=False, indent=2)

print('Model bilgisi kaydedildi: model_bilgisi.json')
print(json.dumps(model_bilgisi, ensure_ascii=False, indent=2))

**Deploy için kullanım notu:** Kaydedilen `maas_tahmin_modeli.joblib` dosyası, `joblib.load()` ile başka bir Python ortamında (örn. bir FastAPI/Flask API içinde) yüklenip `model.predict(yeni_veri)` şeklinde çağrılabilir. `yeni_veri`, `model_bilgisi.json` içindeki `ozellik_sirasi` listesindeki sütunlarla aynı sırada ve isimde bir `pandas.DataFrame` olmalıdır.

## Sonuç ve Çıkarım

Bu proje, Stack Overflow Developer Survey 2025 verisiyle bir yazılımcının profilinden aylık maaşını tahmin eden optimize edilmiş bir XGBoost regresyon modeli ile kişisel tercihlerine göre 9 yazılım alanı arasından öneri sunan adil (normalize edilmiş) bir puanlama sistemini birleştiriyor. İkisi birlikte, kariyer yönü belirlemeye çalışan bir yazılımcıya hem objektif piyasa verisi hem de kişisel eğilim bazlı bir karar desteği sunuyor.

## Kontroller

- [x] Kaggle verisi yerelden okundu veya otomatik indirildi
- [x] Veri boyutu ve sütunlar doğrulandı (49.123 satır × 170 sütun ham veri)
- [x] Eksik değer ve uç değer (outlier) temizliği yapıldı
- [x] Sayısal ve kategorik sütunlar tek bir model boru hattında (Pipeline) işlendi
- [x] Eğitim/test ayrımı yapıldı (%80/%20)
- [x] 4 farklı model karşılaştırıldı (LinearRegression, RandomForest, GradientBoosting, XGBoost)
- [x] XGBoost, GridSearch ile hiperparametre optimizasyonundan geçirildi
- [x] R² ve MAE metrikleri hesaplandı ve yorumlandı
- [x] Model özellik önemi (feature importance) görselleştirildi
- [x] Yeni profiller için çalışan tahmin fonksiyonu yazıldı (USD + TL)
- [x] 9 alanlı, 15 soruluk, adil (normalize) puanlamalı alan önerici tamamlandı
- [x] Model deploy için dışa aktarıldı (.joblib + .json)
- [x] Sınırlamalar ve dürüst değerlendirme notu eklendi

## Sonraki Adımlar

1. Modeli FastAPI/Flask ile bir API'ye taşıyıp frontend'e bağla.
2. Güncel döviz kuru API'siyle sabit kur varsayımını dinamik hale getir.
3. Alan önerici anketini modele bağlayıp önerilen alana göre otomatik maaş tahmini gösterecek şekilde birleştir.
4. Veri setini güncel tutmak için yıllık Stack Overflow anketleriyle modeli periyodik olarak yeniden eğit.
5. GitHub reposuna README, kurulum talimatları ve canlı demo linki ekle.

**Son söz:** Bu model, kesin bir maaş garantisi vermez; sınırlı sayıda özellikle eğitilmiş, sistematik olarak karşılaştırılmış ve optimize edilmiş, dürüstçe değerlendirilmiş bir tahmin aracıdır. Kariyer kararı verirken tek başına değil, bir referans noktası olarak kullanılmalıdır.

---
*Software Persona Stajı — 5 Günlük Yapay Zeka Eğitimi Bitirme Projesi*